# FairVGNN — Mitigating Sensitive Attribute Leakage in Graph Neural Networks

This notebook implements the core idea of [Improving Fairness in Graph Neural Networks via Mitigating Sensitive Attribute Leakage](https://arxiv.org/abs/2206.03426) (Wang, Zhao, Dong, Chen, Li & Derr, SIGKDD 2022), on the Pokec-z social network dataset.

**Read `agarwal.ipynb` in the CDEI-Bias-Mitigation repository before this notebook.** That notebook derives fair classification as a constrained optimisation problem solved by a two-player min-max game between a *learner* and a *regulator*, with the regulator's move being an explicit Lagrange multiplier update. FairVGNN plays exactly the same kind of game, with one change: the regulator is replaced by a trained neural network (a **discriminator**) that adversarially penalises the encoder for leaking the sensitive attribute, and a third player (a **generator**) is added, which learns to suppress the input feature channels most responsible for that leakage.

## Motivation

Graph neural networks aggregate a node's features with its neighbours' features (message passing / feature propagation). The paper's central empirical claim is that this propagation step can *increase* the correlation between ordinary, innocuous-looking features and the sensitive attribute — even if the sensitive attribute itself is excluded from the model's inputs, propagation can smuggle it back in through correlated proxies. We verify this claim directly on real data below before building any fairness mechanism.

## The three-player game

| Player | Role | Analogue in `agarwal.ipynb` |
| --- | --- | --- |
| **Generator** | Learns a per-feature "keep" mask, trying to suppress channels that leak the sensitive attribute while keeping the rest | — (new; there is no equivalent for tabular reductions) |
| **Encoder + classifier** ("the learner") | Predicts the true label from the masked, propagated node features | The base classifier being re-weighted by $\lambda$ |
| **Discriminator** ("the regulator") | Tries to predict the sensitive attribute from the encoder's output embeddings | The Lagrange multiplier $\lambda$, but now a trained network instead of a closed-form update |

Each training epoch alternates four steps: train the discriminator to *detect* leakage, train the classifier to predict the label, train the generator to *fool* the discriminator (while a regularisation term stops it masking everything), then clip the encoder's first-layer weights in proportion to the learned mask so that suppressed channels genuinely cannot influence the embedding. This is a direct, mechanical generalisation of the min-max idea — the constraint is no longer linear and closed-form, so there is no explicit multiplier to plot, but the alternating best-response structure is identical.

## A note on this notebook vs. the original FairVGNN code

The original research code (`github.com/yuwvandy/FairVGNN`) supports five interchangeable GNN backbones and several loss variants, and depends on `torch-scatter`/`torch-sparse` — compiled C-extension packages that are notoriously version-locked to a specific PyTorch/CUDA build and installed via a separate custom index. This notebook implements the same mechanism with a single GCN backbone, built on PyTorch Geometric's own maintained `GCNConv` layer (the same public API used in the CDEI repo's spirit of keeping dependencies boring), which needs nothing beyond a plain `pip install torch torch_geometric` — no compiled extensions, no CUDA-matched wheel index. This trims runtime for a class session without changing the core algorithm.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.metrics import f1_score, roc_auc_score

torch.manual_seed(0)
np.random.seed(0)

## Load data: Pokec-z (`region_job`)

Pokec is a Slovak social network; this preprocessed slice (from the [FairGNN](https://github.com/EnyanDai/FairGNN) benchmark) uses **region** (which of two broad geographic regions a user is in) as the sensitive attribute and a binarised **job field** as the label. Node features are ~275 lifestyle/demographic indicator variables (hobbies, music taste, education, etc.) — none of them is the sensitive attribute itself.

In [3]:
DATA_DIR = "data"


def load_pokec(data_dir, label_number=1000):
    df = pd.read_csv(f"{data_dir}/region_job.csv")
    sens_attr, predict_attr = "region", "I_am_working_in_field"

    # The predict attribute has several sparse categories plus a "-1" missing
    # code; we keep only the two most common *valid* categories (0 and 1),
    # matching the standard "region_job" benchmark split.
    df = df[df[predict_attr].isin([0, 1])].copy().reset_index(drop=True)

    labels = df[predict_attr].values.astype(np.int64)
    sens = df[sens_attr].values.astype(np.int64)
    feature_cols = [c for c in df.columns if c not in ["user_id", sens_attr, predict_attr]]
    features = df[feature_cols].values.astype(np.float32)

    user_ids = df["user_id"].values
    id_to_idx = {uid: i for i, uid in enumerate(user_ids)}

    edges_df = pd.read_csv(f"{data_dir}/region_job_relationship.txt", sep="\t", header=None)
    edges = edges_df.values
    keep = np.isin(edges[:, 0], user_ids) & np.isin(edges[:, 1], user_ids)
    edges = edges[keep]
    edge_index = np.array([[id_to_idx[a], id_to_idx[b]] for a, b in edges]).T
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)  # undirected

    features = torch.tensor(features)
    fmin, fmax = features.min(dim=0).values, features.max(dim=0).values
    denom = fmax - fmin
    denom[denom == 0] = 1
    features = 2 * (features - fmin) / denom - 1  # scale to [-1, 1]

    labels = torch.tensor(labels, dtype=torch.long)
    sens = torch.tensor(sens, dtype=torch.long)

    rng = np.random.default_rng(20)
    idx0, idx1 = np.where(labels.numpy() == 0)[0], np.where(labels.numpy() == 1)[0]
    rng.shuffle(idx0)
    rng.shuffle(idx1)
    n_train = min(int(0.5 * min(len(idx0), len(idx1))), label_number // 2)
    idx_train = np.concatenate([idx0[:n_train], idx1[:n_train]])
    idx_val = np.concatenate([idx0[n_train:int(1.5 * n_train)], idx1[n_train:int(1.5 * n_train)]])
    idx_test = np.concatenate([idx0[int(1.5 * n_train):], idx1[int(1.5 * n_train):]])

    def to_mask(idx):
        m = torch.zeros(features.shape[0], dtype=torch.bool)
        m[torch.tensor(idx, dtype=torch.long)] = True
        return m

    return Data(x=features, edge_index=edge_index, y=labels.float(), sens=sens,
                train_mask=to_mask(idx_train), val_mask=to_mask(idx_val), test_mask=to_mask(idx_test))


data = load_pokec(DATA_DIR, label_number=1000)
print(data)
print(f"train / val / test: {data.train_mask.sum().item()} / {data.val_mask.sum().item()} / {data.test_mask.sum().item()}")
print(f"P(y=1): {data.y.mean():.3f}   P(sens=1): {data.sens.float().mean():.3f}")

Data(x=[6728, 276], edge_index=[2, 27992], y=[6728], sens=[6728], train_mask=[6728], val_mask=[6728], test_mask=[6728])
train / val / test: 1000 / 500 / 5228
P(y=1): 0.292   P(sens=1): 0.356


## Verifying the motivation: does propagation increase sensitive-attribute leakage?

We compare, for every feature channel, its correlation with the sensitive attribute before and after one round of graph propagation (a single `GCNConv` with its weights fixed to the identity, so it purely averages each node's features with its neighbours' — no learning involved yet).

In [4]:
X = data.x.numpy()
sens_np = data.sens.numpy()

corr_raw = np.nan_to_num(np.array([
    abs(np.corrcoef(X[:, i], sens_np)[0, 1]) for i in range(X.shape[1])
]))

identity_conv = GCNConv(X.shape[1], X.shape[1], bias=False)
with torch.no_grad():
    identity_conv.lin.weight.copy_(torch.eye(X.shape[1]))
propagated = identity_conv(data.x, data.edge_index).detach().numpy()

corr_propagated = np.nan_to_num(np.array([
    abs(np.corrcoef(propagated[:, i], sens_np)[0, 1]) for i in range(propagated.shape[1])
]))

print(f"Mean |correlation with sensitive attribute|  raw features: {corr_raw.mean():.4f}")
print(f"Mean |correlation with sensitive attribute|  after propagation: {corr_propagated.mean():.4f}")
print(f"Max  |correlation with sensitive attribute|  raw features: {corr_raw.max():.4f}")
print(f"Max  |correlation with sensitive attribute|  after propagation: {corr_propagated.max():.4f}")

Mean |correlation with sensitive attribute|  raw features: 0.0160
Mean |correlation with sensitive attribute|  after propagation: 0.0201
Max  |correlation with sensitive attribute|  raw features: 0.1300
Max  |correlation with sensitive attribute|  after propagation: 0.1721


Propagation measurably increases correlation with the sensitive attribute, on average and at the most-affected channel — the leakage mechanism the paper targets is real on this data, even though none of the raw features *is* the sensitive attribute.

## Model components

- `ChannelMasker` — the **generator**: one trainable 2-way logit per input feature, turned into a soft 0/1 "keep" mask via the Gumbel-softmax trick (so it's differentiable).
- `GCNEncoder` — a single `GCNConv` layer producing node embeddings, with a `clip_parameters` method used by the weight-clipping step.
- `MLPHead` — a single linear layer, reused for both the **classifier** (predicts the label) and the **discriminator** (predicts the sensitive attribute).

In [5]:
class ChannelMasker(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.weights = nn.Parameter(torch.distributions.Uniform(0, 1).sample((num_features, 2)))

    def forward(self):
        return self.weights


class GCNEncoder(nn.Module):
    def __init__(self, num_features, hidden):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden, bias=True)

    def clip_parameters(self, channel_weights, clip_e=1.0):
        """Clip each input-feature column of the first layer's weight matrix
        in proportion to how much the generator currently trusts that channel.
        A channel with keep-weight near 0 ends up with a near-zero weight column,
        so it genuinely cannot influence the embedding regardless of what the
        soft Gumbel mask let through upstream."""
        w = self.conv1.lin.weight.data
        for i in range(w.shape[1]):
            w[:, i].clamp_(-clip_e * channel_weights[i], clip_e * channel_weights[i])

    def forward(self, x, edge_index):
        return self.conv1(x, edge_index)


class MLPHead(nn.Module):
    def __init__(self, hidden, out=1):
        super().__init__()
        self.lin = Linear(hidden, out)

    def forward(self, h):
        return self.lin(h)


def fair_metric(pred, labels, sens):
    """Statistical parity and equal opportunity gap, as in the original paper."""
    idx_s0, idx_s1 = sens == 0, sens == 1
    idx_s0_y1, idx_s1_y1 = idx_s0 & (labels == 1), idx_s1 & (labels == 1)
    parity = abs(pred[idx_s0].mean() - pred[idx_s1].mean())
    equality = abs(pred[idx_s0_y1].mean() - pred[idx_s1_y1].mean())
    return parity, equality


def evaluate(encoder, classifier, data, mask_vec=None):
    encoder.eval()
    classifier.eval()
    with torch.no_grad():
        x = data.x * mask_vec if mask_vec is not None else data.x
        h = encoder(x, data.edge_index)
        probs = torch.sigmoid(classifier(h).squeeze())
        pred = (probs > 0.5).long()

        test_y = data.y[data.test_mask].long()
        test_pred = pred[data.test_mask]
        acc = (test_pred == test_y).float().mean().item()
        f1 = f1_score(test_y.numpy(), test_pred.numpy())
        auc = roc_auc_score(test_y.numpy(), probs[data.test_mask].numpy())
        parity, equality = fair_metric(
            test_pred.numpy(), test_y.numpy(), data.sens[data.test_mask].numpy()
        )
    return acc, f1, auc, parity, equality


NUM_FEATURES = data.x.shape[1]
HIDDEN = 64

## Baseline: plain GCN, no fairness intervention

A single-layer GCN encoder and classifier trained the ordinary way, for comparison.

In [6]:
torch.manual_seed(0)
enc_bl = GCNEncoder(NUM_FEATURES, HIDDEN)
clf_bl = MLPHead(HIDDEN)
opt_bl = torch.optim.Adam(list(enc_bl.parameters()) + list(clf_bl.parameters()), lr=0.01, weight_decay=1e-5)

t0 = time.time()
for epoch in range(100):
    enc_bl.train()
    clf_bl.train()
    opt_bl.zero_grad()
    h = enc_bl(data.x, data.edge_index)
    logits = clf_bl(h).squeeze()
    loss = F.binary_cross_entropy_with_logits(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    opt_bl.step()
print(f"Baseline training time: {time.time() - t0:.1f}s")

bl_acc, bl_f1, bl_auc, bl_parity, bl_equality = evaluate(enc_bl, clf_bl, data)
print(f"Baseline  accuracy={bl_acc:.3f}  F1={bl_f1:.3f}  AUC={bl_auc:.3f}  "
      f"parity={bl_parity:.3f}  equality={bl_equality:.3f}")

Baseline training time: 1.0s
Baseline  accuracy=0.842  F1=0.713  AUC=0.917  parity=0.037  equality=0.011


## The adversarial game, live

Each epoch: train the discriminator to detect the sensitive attribute from the (currently masked) embedding, train the classifier on the true labels, train the generator to fool the discriminator while staying close to "keep everything", then clip the encoder's weights by the learned mask. `K` controls how many independent Gumbel-softmax mask samples are averaged per step (reduces variance from the stochastic masking).

In [7]:
torch.manual_seed(0)
generator = ChannelMasker(NUM_FEATURES)
encoder = GCNEncoder(NUM_FEATURES, HIDDEN)
classifier = MLPHead(HIDDEN)
discriminator = MLPHead(HIDDEN)

opt_g = torch.optim.Adam(generator.parameters(), lr=0.001)
opt_e = torch.optim.Adam(encoder.parameters(), lr=0.001, weight_decay=1e-5)
opt_c = torch.optim.Adam(classifier.parameters(), lr=0.001, weight_decay=1e-5)
opt_d = torch.optim.Adam(discriminator.parameters(), lr=0.001)

K = 5            # Gumbel-softmax samples averaged per step
RATIO = 1.0      # weight on the "don't mask everything" regulariser
EPOCHS = 100

t0 = time.time()
for epoch in range(EPOCHS):
    feature_weights = generator()
    masks = [F.gumbel_softmax(feature_weights, tau=1, hard=False)[:, 0] for _ in range(K)]

    # 1. discriminator step: learn to detect the sensitive attribute
    discriminator.train(); encoder.train()
    opt_d.zero_grad(); opt_e.zero_grad()
    loss_d = sum(
        F.binary_cross_entropy_with_logits(
            discriminator(encoder(data.x * masks[k].detach(), data.edge_index)).squeeze(),
            data.sens.float(),
        )
        for k in range(K)
    ) / K
    loss_d.backward()
    opt_d.step(); opt_e.step()

    # 2. classifier step: learn to predict the label
    classifier.train(); encoder.train()
    opt_c.zero_grad(); opt_e.zero_grad()
    loss_c = sum(
        F.binary_cross_entropy_with_logits(
            classifier(encoder(data.x * masks[k].detach(), data.edge_index)).squeeze()[data.train_mask],
            data.y[data.train_mask],
        )
        for k in range(K)
    ) / K
    loss_c.backward()
    opt_e.step(); opt_c.step()

    # 3. generator step: fool the discriminator, don't mask everything
    generator.train(); encoder.train(); discriminator.eval()
    opt_g.zero_grad(); opt_e.zero_grad()
    feature_weights = generator()
    loss_g = 0
    for _ in range(K):
        mask = F.gumbel_softmax(feature_weights, tau=1, hard=False)[:, 0]
        h = encoder(data.x * mask, data.edge_index)
        out = torch.sigmoid(discriminator(h).squeeze())
        loss_g = loss_g + F.mse_loss(out, 0.5 * torch.ones_like(out)) \
                         + RATIO * F.mse_loss(mask, torch.ones_like(mask))
    (loss_g / K).backward()
    opt_g.step(); opt_e.step()

    # 4. weight clip: bake the learned mask into the encoder's weights
    with torch.no_grad():
        avg_mask = torch.stack(masks).mean(dim=0)
        encoder.clip_parameters(avg_mask, clip_e=1.0)

print(f"FairVGNN training time: {time.time() - t0:.1f}s")

FairVGNN training time: 16.5s


### Evaluate FairVGNN and compare to the baseline

In [8]:
with torch.no_grad():
    final_mask = torch.stack(
        [F.gumbel_softmax(generator(), tau=1, hard=False)[:, 0] for _ in range(K)]
    ).mean(dim=0)

acc, f1, auc, parity, equality = evaluate(encoder, classifier, data, mask_vec=final_mask)

print(f"{'Metric':<12}{'Baseline':>12}{'FairVGNN':>12}")
print(f"{'Accuracy':<12}{bl_acc:>12.3f}{acc:>12.3f}")
print(f"{'F1':<12}{bl_f1:>12.3f}{f1:>12.3f}")
print(f"{'AUC':<12}{bl_auc:>12.3f}{auc:>12.3f}")
print(f"{'Parity':<12}{bl_parity:>12.3f}{parity:>12.3f}")
print(f"{'Equality':<12}{bl_equality:>12.3f}{equality:>12.3f}")

Metric          Baseline    FairVGNN
Accuracy           0.842       0.824
F1                 0.713       0.701
AUC                0.917       0.920
Parity             0.037       0.039
Equality           0.011       0.025


## Checking the mechanism directly: did the generator learn to suppress leaky channels?

Rather than rely only on the downstream parity/equality numbers (which are noisy on a single run — see the discussion below), we can check the mechanism itself: does the generator's learned keep-probability for a channel relate to that channel's correlation with the sensitive attribute?

In [9]:
with torch.no_grad():
    keep_prob = F.softmax(generator(), dim=1)[:, 0].numpy()

mechanism_corr = np.corrcoef(keep_prob, corr_raw)[0, 1]
print(f"Correlation between (learned keep-probability) and (channel's |correlation with sens|): {mechanism_corr:.3f}")

order = np.argsort(keep_prob)
most_suppressed, most_retained = order[:20], order[-20:]
print(f"Mean |corr with sens| — 20 most-suppressed channels: {corr_raw[most_suppressed].mean():.4f}")
print(f"Mean |corr with sens| — 20 most-retained channels:   {corr_raw[most_retained].mean():.4f}")
print(f"Mean |corr with sens| — all channels:                {corr_raw.mean():.4f}")

Correlation between (learned keep-probability) and (channel's |correlation with sens|): -0.106
Mean |corr with sens| — 20 most-suppressed channels: 0.0155
Mean |corr with sens| — 20 most-retained channels:   0.0119
Mean |corr with sens| — all channels:                0.0160


A negative correlation here, and a lower mean for the most-suppressed channels than the most-retained ones, is direct evidence the generator is doing its job — preferentially masking the channels most predictive of the sensitive attribute — independent of whatever noise shows up in the downstream classification metrics.

## Discussion

**On the size of the effect.** Unlike the synthetic `beta=10` experiment in `Causal-Fairness-Auditing`, where the ground-truth effect is exact by construction, this is real, messy social-network data — the baseline demographic parity gap here is modest, and a single training run's parity/equality numbers can be noisy. This mirrors the same clean-synthetic-vs-messy-real contrast that notebook draws between its two experiments, and it's a fair one to make explicit to students: real fairness audits often don't hand you a dramatic before/after, and "the effect is small and noisy" is itself a finding worth reporting honestly rather than a failed demo.

**Discussion questions for students:**

1. Re-run the adversarial training cell with a different `torch.manual_seed(...)`, or wrap it in a loop over 3–5 seeds and report mean ± standard deviation for parity and equality, the way you would for any noisy experimental result. Does FairVGNN reliably beat the baseline, or is the single run above within noise?
2. `RATIO` controls how strongly the generator is penalised for masking channels. What do you expect happens to accuracy and to the mechanism-level correlation check as `RATIO` grows very large (mask almost nothing) or very small (mask almost everything)?
3. **Connecting back to `agarwal.ipynb`:** both notebooks solve a min-max game with an inner "learner" and an outer "adversary" pushing back on unfairness. What did we gain by replacing the closed-form Lagrange multiplier with a learned discriminator? What did we lose — specifically, what could you read directly off `lambda_vecs_` in the reductions notebook that has no clean analogue here?